In [1]:
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo trl peft accelerate bitsandbytes xformers

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
import gc
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 1. Configurations

In [4]:
# Model
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True

# Output
OUTPUT_DIR = "/kaggle/working/qwen2_5_1_5b_unsloth_bank_support_lora"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Data split
TRAIN_SIZE = 0.85
VAL_SIZE_FROM_TEMP = 0.50

# Training
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

# Inference settings after training
INFERENCE_MAX_NEW_TOKENS = 170
INFERENCE_TEMPERATURE = 0.40
INFERENCE_TOP_P = 0.90

print("Output directory:", OUTPUT_DIR)

Output directory: /kaggle/working/qwen2_5_1_5b_unsloth_bank_support_lora


## 2. Load Synthesized Dataset

In [5]:
DATA_DIR = "/kaggle/input/datasets/coolguy312/1k-complaint-synthetic-response-llama3-18b/Complaint_SyntheticResponse_1k.csv"

synth_df = pd.read_csv(DATA_DIR, low_memory=False)
print("DF shape:", synth_df.shape)
print("Columns:", synth_df.columns.tolist())
synth_df.head()

DF shape: (650, 3)
Columns: ['narrative', 'product', 'response']


,narrative,product,response
0,recently sent lvnv funding llc letter disputin...,debt_collection,We have received your letter disputing the all...
1,account open yet report closed stated falsely ...,credit_reporting,We are reviewing your complaint regarding the ...
2,account account account belong mean person acc...,credit_reporting,We understand that you're concerned about the ...
3,paid charged account full kohl impression woul...,debt_collection,We understand that you're concerned about the ...
4,account open yet report closed stated falsely ...,credit_reporting,We have received your complaint regarding the ...


## 3. Clean and filter synthesized responses

Removes failures observed during synthesis, including:
- Template generic openings like “Dear valued customer”
- Asking customers for account numbers or additional documents
- Placeholders like `[Creditor's Name]`
- Claims that the agent already reviewed the case
- Markdown, bullets, or numbered lists

In [6]:
synth_df = synth_df[["narrative", "product", "response"]].copy()
synth_df = synth_df.dropna(subset=["narrative", "response"]).copy()

for col in ["narrative", "product", "response"]:
    synth_df[col] = synth_df[col].astype(str).str.strip()

# Remove empty rows and duplicates
synth_df = synth_df[(synth_df["narrative"].str.len() > 0) & (synth_df["response"].str.len() > 0)].copy()
synth_df = synth_df.drop_duplicates(subset=["narrative", "response"]).copy()
print("DF shape after dropping NAs and duplicates:", synth_df.shape)


def clean_response(text: str) -> str:
    text = str(text or "").strip()
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r'[`*_#>"]', "", text)
    text = re.sub(r"^(response:|assistant:)", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"\s+", " ", text).strip()

    # Trim to the last complete sentence if possible
    last_end = max(text.rfind("."), text.rfind("!"), text.rfind("?"))
    if last_end > 80:
        text = text[:last_end + 1].strip()

    return text


def is_valid_response(text: str) -> bool:
    text = str(text or "").strip()
    lower = text.lower()
    words = text.split()

    # Length filtering
    if len(words) < 45 or len(words) > 190:
        return False

    bad_patterns = [
        "dear valued customer",
        "thank you for reaching out",
        "thank you for your feedback",
        "account number",
        "phone number",
        "email address",
        "support@example",
        "could you please provide",
        "please provide",
        "please share",
        "kindly provide",
        "contact us at",
        "we've reviewed",
        "we have reviewed",
        "we reviewed",
        "i'm reviewing",
        "i am reviewing",
        "i reviewed",
        "personally reviewed",
        "after reviewing",
        "company name",
        "creditor's name",
        "placeholder",
        "[",
        "]",
        "unable to validate the debt",
    ]

    if any(p in lower for p in bad_patterns):
        return False

    # Reject markdown bullets, markdown bold, and numbered-list style outputs
    if ("**" in text or re.search(r"\s*[-*]\s+", text) or re.search(r"\s*\d+\.\s+", text)):
        return False

    if not text or text[-1] not in {".", "!", "?", '"', "'"}:
        return False

    return True


synth_df["response"] = synth_df["response"].apply(clean_response)

before = len(synth_df)
valid_mask = synth_df["response"].apply(is_valid_response)
synth_df = synth_df[valid_mask].reset_index(drop=True)
print(f"Removed {before - len(synth_df)} low-quality responses")
print("Final clean samples:", len(synth_df))

synth_df.sample(min(5, len(synth_df)), random_state=SEED)[["narrative", "product", "response"]]

After null/empty/dedup: (650, 3)
Removed 42 low-quality responses
Final clean samples: 608


,narrative,product,response
109,mortgage paid date loancare sent lien release ...,mortgages_and_loans,We understand that you're concerned about the ...
10,notice alleged debt ncb management service owe...,debt_collection,We have taken note of your concern regarding a...
184,wrote u department education concerning credit...,credit_reporting,We have received your complaint regarding a di...
77,chase upgraded private client sudden saying av...,retail_banking,I understand that you're concerned about the r...
538,received voicemail citibank suspicious activit...,credit_card,We understand that you've experienced numerous...


## 4. Train / validation / test split

In [7]:
train_df, temp_df = train_test_split(
    synth_df,
    train_size=TRAIN_SIZE,
    random_state=SEED,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=VAL_SIZE_FROM_TEMP,
    random_state=SEED,
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape: ", train_df.shape)
print("Val shape: ", val_df.shape)
print("Test shape: ", test_df.shape)

Train shape:  (516, 3)
Val shape:  (46, 3)
Test shape:  (46, 3)


## 5. Load Qwen2.5-1.5B with Unsloth

Base model loaded in 4-bit quantization. The LoRA adapter is PEFT, LoRA, and QLoRA

In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

# Disable any hard-coded max length cap
# Generation length is controlled manually
model.generation_config.max_length = None
model.config.max_length = None

# Set right padding to ensure padding tokens are added at the end of input
tokenizer.padding_side = "right"

# Assign End of Sequence Token if there is no tokens
# Ensures model can handle padded inputs and prevents error in batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

IM_END_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
print("IM_END_TOKEN_ID:", IM_END_TOKEN_ID)
print("pad_token_id:", tokenizer.pad_token_id)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
IM_END_TOKEN_ID: 151645
pad_token_id: 151665


## 6. Format examples for supervised fine-tuning (SFT)

In [9]:
TRAIN_SYSTEM_PROMPT = (
    "You are a banking customer support specialist responding to a logged complaint. "
    "Write one professional paragraph that acknowledges the customer's specific issue, "
    "uses only information from the complaint, and explains what the support team will review internally. "
    "Do not ask the customer for account numbers, documents, dates, or additional information. "
    "Do not include phone numbers, emails, markdown, bullet points, or numbered lists. "
    "Do not promise a specific outcome."
)


def build_train_messages(row):
    product = str(row.get("product", "")).strip()
    complaint = str(row["narrative"]).strip()

    if product:
        user_content = f"Product category:\n{product} \n\n Customer complaint:\n{complaint}"
    else:
        user_content = f"Customer complaint:{complaint}"

    return [
        {"role": "system", "content": TRAIN_SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": str(row.get("response", "")).strip()},
    ]


def to_text(row):
    return tokenizer.apply_chat_template(
        build_train_messages(row),
        tokenize=False,
        add_generation_prompt=False,
    )

print(to_text(train_df.iloc[0].to_dict())[:1200])

<|im_start|>system
You are a banking customer support specialist responding to a logged complaint. Write one professional paragraph that acknowledges the customer's specific issue, uses only information from the complaint, and explains what the support team will review internally. Do not ask the customer for account numbers, documents, dates, or additional information. Do not include phone numbers, emails, markdown, bullet points, or numbered lists. Do not promise a specific outcome.<|im_end|>
<|im_start|>user
Product category:
debt_collection 

 Customer complaint:
currently receiving debt collection letter false negative collection report credit report filed resurgent receivables llc resurgent claiming owe debt amount dating back identity stolen back investigating charge account determined fact victim fraud debt cleared account posse letter dated stating fraudulent information would deleted credit report updated information would reported credit bureau matter closed six year ago stil

In [10]:
# Convert pandas DataFrames into HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

# Formats each complaint-response pair into a single training string
def formatting_prompts_func(examples):
    texts = []
    for i in range(len(examples["narrative"])):
        row = {k: examples[k][i] for k in examples.keys()}
        texts.append(to_text(row))
    return {"text": texts}

# Apply the formatting function to the training/val/test dataset
train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_dataset.column_names,
)

val_dataset = val_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=val_dataset.column_names,
)

test_dataset = test_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=test_dataset.column_names,
)

print(train_dataset)
print(train_dataset[0]["text"][:1000 # Preview one formatted training example

Map:   0%|          | 0/516 [00:00<?, ? examples/s]

Map:   0%|          | 0/46 [00:00<?, ? examples/s]

Map:   0%|          | 0/46 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 516
})
<|im_start|>system
You are a banking customer support specialist responding to a logged complaint. Write one professional paragraph that acknowledges the customer's specific issue, uses only information from the complaint, and explains what the support team will review internally. Do not ask the customer for account numbers, documents, dates, or additional information. Do not include phone numbers, emails, markdown, bullet points, or numbered lists. Do not promise a specific outcome.<|im_end|>
<|im_start|>user
Product category:
debt_collection 

 Customer complaint:
currently receiving debt collection letter false negative collection report credit report filed resurgent receivables llc resurgent claiming owe debt amount dating back identity stolen back investigating charge account determined fact victim fraud debt cleared account posse letter dated stating fraudulent information would deleted credit report updated information would

## 7. Add LoRA adapters

In [11]:
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, # Rank of LoRA Adapter, smaller = faster

    # modules = which parts of transformer to modify
    # - q_proj, k_proj, v_proj: attention mechanism (How tokens interact)
    # - o_proj: output of attention
    # - gate_proj, up_proj, down_proj: feedforward network (Reasoning)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32, # Scaling factor for LoRA updates, usually 2 x r
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=True,
    loftq_config=None,
)

model.print_trainable_parameters()

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 8. Train with SFTTrainer

In [12]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,

    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

# Mask system and user tokens from the loss, training only on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user",
    response_part="<|im_start|>assistant",
)

train_result = trainer.train()
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/516 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/516 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/46 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=8):   0%|          | 0/46 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 448 | Num Epochs = 1 | Total steps = 28
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.017804,0.987364


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen2_5_1_5b_unsloth_bank_support_lora/checkpoint-28/tokenizer_config.json.


TrainOutput(global_step=28, training_loss=1.115203091076442, metrics={'train_runtime': 103.7193, 'train_samples_per_second': 4.319, 'train_steps_per_second': 0.27, 'total_flos': 1296813057767424.0, 'train_loss': 1.115203091076442, 'epoch': 1.0})

## 9. Evaluate and preview generation

In [13]:
history = trainer.state.log_history

eval_logs = [log for log in history if "eval_loss" in log]
eval_logs

[{'eval_loss': 0.9873642325401306,
  'eval_runtime': 3.6189,
  'eval_samples_per_second': 11.882,
  'eval_steps_per_second': 3.04,
  'epoch': 1.0,
  'step': 28}]

In [17]:
FastLanguageModel.for_inference(model)


def generate_support_response(complaint: str, product: str = "") -> str:
    row = {"narrative": complaint, "product": product, "response": ""}
    messages = [m for m in build_train_messages(row) if m["role"] != "assistant"]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=INFERENCE_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=INFERENCE_TEMPERATURE,
            top_p=INFERENCE_TOP_P,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[tokenizer.eos_token_id, IM_END_TOKEN_ID],
            use_cache=True,
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


sample = test_df.iloc[0]
print("Complaint:")
print(sample["narrative"], "\n")
print("Fine-tuned response:")
print(generate_support_response(sample["narrative"], sample.get("product", "")),"\n")
print("Target response:")
print(sample["response"])

Complaint:
respond phone call email regarding incorrect information filed credit bureau filed collection nothing construction company md someone else used publicly available info business resulting negative info personal credit profile 

Fine-tuned response:
We understand that you're concerned about an error in your credit report due to incorrect information being submitted by another party on behalf of a construction company based in Maryland. You've reported this issue to both our internal systems and external agencies such as the Federal Trade Commission (FTC) and have received confirmation that no collections were made against your account. Our team will investigate further into how this discrepancy occurred and ensure that any necessary corrections are made to reflect accurate financial history on your record. We'll also verify with relevant parties involved in the transaction to confirm their actions align with industry standards and regulations governing consumer reporting pract

## 10. Save adapter and push merged model to Hugging Face

In [ ]:
# Save LoRA adapter.
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter to", OUTPUT_DIR)

# Save merged model for deployment.
MERGED_DIR = OUTPUT_DIR + "_merged"

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved to", MERGED_DIR)

# Hugging Face Hub configurations
PUSH_TO_HUB = True
HF_REPO = "Coolguy41234/qwen-bank-support"
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

if PUSH_TO_HUB: 
    model.push_to_hub_merged(
            HF_REPO,
            tokenizer,
            save_method="merged_16bit",
            token=HF_TOKEN,
        )
    print(f"Pushed merged model to: https://huggingface.co/{HF_REPO}")

# Zip adapter for local download
!cd /kaggle/working && zip -qr qwen2_5_1_5b_unsloth_bank_support_lora.zip qwen2_5_1_5b_unsloth_bank_support_lora

Saved adapter to /kaggle/working/qwen2_5_1_5b_unsloth_bank_support_lora
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 4405.78it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:19<00:00, 19.61s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen2_5_1_5b_unsloth_bank_support_lora_merged`
Merged model saved to /kaggle/working/qwen2_5_1_5b_unsloth_bank_support_lora_merged


...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:49<00:00, 49.57s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Coolguy41234/qwen-bank-support`
Pushed merged model to: https://huggingface.co/Coolguy41234/qwen-bank-support
Zip ready: /kaggle/working/qwen2_5_1_5b_unsloth_bank_support_lora.zip
